# Lab 1 – Text Dataset Loading, Tokenization & DataLoader

**Modifications from original:**
- Dataset: `ag_news` instead of WikiText-2
- Tokenizer: `distilbert-base-uncased` instead of GPT-2
- Block size: 256 tokens instead of 128
- Added: vocab size info, decode sanity check, attention mask stats

In [1]:
!pip install datasets

In [2]:
!pip install transformers datasets torch

In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import DataLoader
import torch

In [4]:
# 1. Load AG News dataset (news topic classification — 4 categories, ~120k training articles)
# We only use the 'text' field for language-model-style tokenization
dataset = load_dataset("ag_news", split="train")
print(f"Number of examples in dataset: {len(dataset)}")
print(f"Features: {dataset.features}")
print(f"\nSample text:\n{dataset[0]['text']}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Number of examples in dataset: 120000
Features: {'text': Value('string'), 'label': ClassLabel(names=['World', 'Sports', 'Business', 'Sci/Tech'])}

Sample text:
Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.


In [6]:
# 2. Initialize DistilBERT tokenizer (bidirectional, subword BPE, different from GPT-2's causal BPE)
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
print(f"Tokenizer vocab size: {tokenizer.vocab_size}")
print(f"Special tokens: PAD={tokenizer.pad_token!r}, CLS={tokenizer.cls_token!r}, SEP={tokenizer.sep_token!r}")

Tokenizer vocab size: 30522
Special tokens: PAD='[PAD]', CLS='[CLS]', SEP='[SEP]'


In [12]:
# 3. Tokenize using batched .map() — strip labels, keep only text
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=False,
        return_special_tokens_mask=False,
        return_token_type_ids=False  # ← this stops DistilBERT from generating it at all
    )

tokenized_ds = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text", "label"]
)

first_ids = tokenized_ds[0]["input_ids"]
print(f"Columns in dataset: {tokenized_ds.column_names}")  # should only show: input_ids, attention_mask
print(f"First example token count: {len(first_ids)}")
print(f"First 20 token IDs: {first_ids[:20]}")
print(f"Decoded back: {tokenizer.decode(first_ids[:20])}")

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Columns in dataset: ['input_ids', 'attention_mask']
First example token count: 41
First 20 token IDs: [101, 2813, 2358, 1012, 6468, 15020, 2067, 2046, 1996, 2304, 1006, 26665, 1007, 26665, 1011, 2460, 1011, 19041, 1010, 2813]
Decoded back: [CLS] wall st. bears claw back into the black ( reuters ) reuters - short - sellers, wall


In [13]:
# 4. Group into fixed-length blocks of 256 tokens
block_size = 256

def group_texts(examples):
    concatenated_inputs = sum(examples["input_ids"], [])
    concatenated_masks  = sum(examples["attention_mask"], [])

    total_len = (len(concatenated_inputs) // block_size) * block_size
    concatenated_inputs = concatenated_inputs[:total_len]
    concatenated_masks  = concatenated_masks[:total_len]

    result_input_ids = [concatenated_inputs[i:i+block_size] for i in range(0, total_len, block_size)]
    result_masks     = [concatenated_masks[i:i+block_size]  for i in range(0, total_len, block_size)]

    return {"input_ids": result_input_ids, "attention_mask": result_masks}

lm_ds = tokenized_ds.map(group_texts, batched=True, batch_size=1000)
print(f"Total LM training sequences (block_size={block_size}): {len(lm_ds)}")

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Total LM training sequences (block_size=256): 24866


In [14]:
# 5. Build a DataLoader with a collate function
def collate_fn(batch):
    input_ids = torch.tensor([ex["input_ids"]      for ex in batch], dtype=torch.long)
    attn_mask = torch.tensor([ex["attention_mask"] for ex in batch], dtype=torch.long)
    # For masked/causal LM, labels == input_ids; model handles loss masking internally
    return {"input_ids": input_ids, "attention_mask": attn_mask, "labels": input_ids.clone()}

train_loader = DataLoader(lm_ds, batch_size=8, shuffle=True, collate_fn=collate_fn)
print(f"Number of batches: {len(train_loader)}")

Number of batches: 3109


In [15]:
# 6. Verify batch shapes and inspect attention mask statistics
for batch in train_loader:
    ids   = batch["input_ids"]
    masks = batch["attention_mask"]
    labels = batch["labels"]

    print(f"input_ids shape : {ids.shape}")
    print(f"attention_mask  : {masks.shape}")
    print(f"labels shape    : {labels.shape}")

    # Extra: fraction of real tokens vs padding in this batch
    real_tokens = masks.sum().item()
    total_tokens = masks.numel()
    print(f"Real-token ratio: {real_tokens}/{total_tokens} = {real_tokens/total_tokens:.2%}")

    # Decode the first sequence of the batch back to readable text
    print(f"\nDecoded sequence 0 (first 50 tokens):\n{tokenizer.decode(ids[0][:50])}")
    break

print("\nDataLoader is working correctly!")

input_ids shape : torch.Size([8, 256])
attention_mask  : torch.Size([8, 256])
labels shape    : torch.Size([8, 256])
Real-token ratio: 2048/2048 = 100.00%

Decoded sequence 0 (first 50 tokens):
largest food group down \ over five percent on wednesday amid concerns about its \ long - term profitability. [SEP] [CLS] weak ice cream sales melt nestle ' s profit zurich ( reuters ) - higher raw material costs and lower ice cream sales in europe ate

DataLoader is working correctly!
